In [2]:
import re
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

df = pd.read_csv("../../data/processed/combined_videos_raw.csv", low_memory=False)

# Re-cast types (same as EDA cell 1)
df["video_publishedAt"] = pd.to_datetime(df["video_publishedAt"], utc=True, errors="coerce")
df["snapshot_utc"]      = pd.to_datetime(df["snapshot_utc"],      utc=True, errors="coerce")

def parse_duration(val):
    if pd.isna(val):
        return np.nan
    m = re.fullmatch(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(val).strip())
    if not m:
        return np.nan
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return float(h * 3600 + mi * 60 + s)

df["duration_sec"] = df["duration"].apply(parse_duration)
df["is_short"]     = df["duration_sec"] <= 60

for col in ["views", "likes", "comments"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 28,037 rows × 56 columns


In [3]:
# Track drop reasons explicitly
drop_log = {}

# 1. Views null (1 row) — no engagement signal at all
mask_no_views = df["views"].isna()
drop_log["views null"] = mask_no_views.sum()

# 2. Duration unparseable (36 rows) — can't classify as short/long
mask_bad_dur = df["duration_sec"].isna()
drop_log["duration unparseable"] = mask_bad_dur.sum()

# 3. Whitespace-only descriptions (2 rows) — treat as null
df.loc[df["video_description"].str.strip().eq(""), "video_description"] = np.nan

drop_mask = mask_no_views | mask_bad_dur
df = df[~drop_mask].reset_index(drop=True)

print("=== Rows dropped ===")
for reason, n in drop_log.items():
    print(f"  {reason:30s}: {n}")
print(f"  {'TOTAL dropped':30s}: {drop_mask.sum()}")
print(f"\nShape after drops: {df.shape}")

=== Rows dropped ===
  views null                    : 1
  duration unparseable          : 36
  TOTAL dropped                 : 37

Shape after drops: (28000, 56)


In [4]:
# Compute before any further filtering so nothing is lost
df["like_rate"]      = df["likes"]    / df["views"].replace(0, np.nan)
df["comment_rate"]   = df["comments"] / df["views"].replace(0, np.nan)
df["views_per_day"]  = df["views"]    / (
    (df["snapshot_utc"] - df["video_publishedAt"]).dt.total_seconds() / 86_400
).replace(0, np.nan)

# Publish date parts — useful for dashboard filters
df["publish_year"]  = df["video_publishedAt"].dt.year
df["publish_month"] = df["video_publishedAt"].dt.to_period("M").dt.to_timestamp()

print("Engagement features added: like_rate, comment_rate, views_per_day")
print(df[["like_rate", "comment_rate", "views_per_day"]].describe().round(4).to_string())

Engagement features added: like_rate, comment_rate, views_per_day
        like_rate  comment_rate  views_per_day
count  27788.0000    27797.0000   2.800000e+04
mean       0.0372        0.0028   2.075305e+04
std        0.0285        0.0070   1.843884e+05
min        0.0000        0.0000   0.000000e+00
25%        0.0226        0.0007   1.679080e+02
50%        0.0338        0.0016   1.011428e+03
75%        0.0471        0.0032   5.259959e+03
max        1.0000        0.4000   1.199053e+07


In [5]:
# Patterns to strip from descriptions (URLs, hashtags, @mentions, emoji, extra whitespace)
URL_RE      = re.compile(r"http\S+|www\.\S+")
HASHTAG_RE  = re.compile(r"#\w+")
MENTION_RE  = re.compile(r"@\w+")
EMOJI_RE    = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F9FF"
    "\U00002700-\U000027BF"
    "\U0001FA00-\U0001FA6F"
    "]+",
    flags=re.UNICODE,
)
PUNCT_RE    = re.compile(r"[^\w\s]")
SPACE_RE    = re.compile(r"\s+")


def clean_text(text: str | None) -> str:
    """Strip noise from a single text field. Returns empty string if null."""
    if pd.isna(text) or not str(text).strip():
        return ""
    text = URL_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = EMOJI_RE.sub(" ", text)
    text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text)
    return text.strip()


df["title_clean"] = df["video_title"].apply(clean_text)
df["desc_clean"]  = df["video_description"].apply(clean_text)

# Tags: pipe-delimited → space-joined cleaned string
df["tags_clean"]  = (
    df["video_tags"]
    .fillna("")
    .apply(lambda x: " ".join(clean_text(tag) for tag in x.split("|") if tag.strip()))
)

print("Cleaned fields: title_clean, desc_clean, tags_clean")
print(f"\nEmpty title_clean  : {(df['title_clean'] == '').sum()}")
print(f"Empty desc_clean   : {(df['desc_clean']  == '').sum()}")
print(f"Empty tags_clean   : {(df['tags_clean']  == '').sum()}")

Cleaned fields: title_clean, desc_clean, tags_clean

Empty title_clean  : 35
Empty desc_clean   : 3052
Empty tags_clean   : 8122


In [6]:
# Strategy:
#   title is doubled to up-weight it vs the longer description
#   tags appended as additional signal where available
#   all three fields are already cleaned

df["bertopic_text"] = (
    df["title_clean"] + " "
    + df["title_clean"] + " "
    + df["desc_clean"]  + " "
    + df["tags_clean"]
).str.strip()

# Collapse any double spaces left from empty fields
df["bertopic_text"] = df["bertopic_text"].str.replace(r"\s+", " ", regex=True).str.strip()

# Token count — proxy for how much signal BERTopic has per doc
df["bertopic_token_count"] = df["bertopic_text"].str.split().str.len()

# Flag docs that are too sparse to be reliable for topic modeling
SPARSE_THRESHOLD = 5
df["is_sparse_text"] = df["bertopic_token_count"] < SPARSE_THRESHOLD

print("=== bertopic_text token count ===")
print(df["bertopic_token_count"].describe().round(1).to_string())
print(f"\nSparse docs (< {SPARSE_THRESHOLD} tokens) : {df['is_sparse_text'].sum():,}")
print()
print("=== Sparse docs breakdown by text source ===")
sparse = df[df["is_sparse_text"]]
print(f"  No description, no tags : {(sparse['desc_clean'].eq('') & sparse['tags_clean'].eq('')).sum()}")
print(f"  Has description         : {sparse['desc_clean'].ne('').sum()}")
print()
print("Sample bertopic_text (research_science):")
print(df[df["category_name"] == "research_science"]["bertopic_text"].iloc[0][:200])
print()
print("Sample bertopic_text (fitness):")
print(df[df["category_name"] == "fitness"]["bertopic_text"].iloc[0][:200])

=== bertopic_text token count ===
count    28000.0
mean       161.8
std        150.5
min          0.0
25%         58.0
50%        118.0
75%        217.0
max       1042.0

Sparse docs (< 5 tokens) : 182

=== Sparse docs breakdown by text source ===
  No description, no tags : 178
  Has description         : 4

Sample bertopic_text (research_science):
The Hidden Engineering of Runways The Hidden Engineering of Runways A few of the things I love about runways Get Nebula using my link for 50 off an annual subscription Watch The Logistics of X Errata 

Sample bertopic_text (fitness):
Why You CAN T Do Pullups Like This Why You CAN T Do Pullups Like This The UPDATED RP HYPERTROPHY APP Become an RP channel member and get instant access to over 200 videos of exclusive in depth trainin


In [7]:
# Drop thumbnail cols, raw text fields (replaced by clean versions),
# internal API IDs not needed downstream, and upload/playlist bookkeeping cols

DROP_COLS = [
    # Thumbnail URLs + dimensions
    "thumb_default_url", "thumb_default_width", "thumb_default_height",
    "thumb_medium_url",  "thumb_medium_width",  "thumb_medium_height",
    "thumb_high_url",    "thumb_high_width",     "thumb_high_height",
    "thumb_standard_url","thumb_standard_width", "thumb_standard_height",
    "thumb_maxres_url",  "thumb_maxres_width",   "thumb_maxres_height",
    # Raw text replaced by clean versions
    "video_description",
    "video_tags",
    # Internal/redundant IDs
    "video_channelId",        # same as channel_id
    "uploads_playlist_id",
    "channel_topicIds",
    "video_topicIds",
    # Low-signal boolean flags rarely used
    "channel_isLinked",
    "channel_madeForKids",
    "projection",             # all standard
    "embeddable",             # all True
]

df_clean = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

print(f"Columns before : {df.shape[1]}")
print(f"Columns after  : {df_clean.shape[1]}")
print()
print("Remaining columns:")
for col in df_clean.columns:
    print(f"  {col}")

Columns before : 67
Columns after  : 42

Remaining columns:
  snapshot_utc
  category_name
  channel_handle_used
  channel_id
  channel_title
  channel_description
  channel_publishedAt
  channel_country
  channel_keywords
  channel_defaultLanguage
  channel_subscriberCount
  channel_viewCount
  channel_videoCount
  channel_topicCategories
  video_id
  video_title
  video_publishedAt
  video_categoryId
  video_defaultLanguage
  video_defaultAudioLanguage
  views
  likes
  comments
  duration
  caption
  licensedContent
  definition
  madeForKids
  video_topicCategories
  duration_sec
  is_short
  like_rate
  comment_rate
  views_per_day
  publish_year
  publish_month
  title_clean
  desc_clean
  tags_clean
  bertopic_text
  bertopic_token_count
  is_sparse_text


In [8]:
# 1. Full cleaned dataset — used by dashboard and modeling
clean_path = "../../data/processed/combined_videos_clean.csv"
df_clean.to_csv(clean_path, index=False)
print(f"Saved combined_videos_clean.csv  — {df_clean.shape[0]:,} rows × {df_clean.shape[1]} cols")

# 2. BERTopic corpus — one document per line, ordered by video_id
# Sparse docs are included but flagged; BERTopic will assign them to outlier topic -1
corpus_path = "../../data/processed/bertopic_corpus.txt"
df_clean["bertopic_text"].to_csv(corpus_path, index=False, header=False)
print(f"Saved bertopic_corpus.txt        — {len(df_clean):,} lines")

# 3. BERTopic metadata — parallel to corpus, same row order
# Everything BERTopic needs to interpret its output
meta_cols = [
    "video_id", "video_title", "channel_title", "category_name",
    "publish_year", "views", "like_rate", "comment_rate",
    "duration_sec", "is_short", "is_sparse_text", "bertopic_token_count",
]
meta_path = "../../data/processed/bertopic_metadata.csv"
df_clean[meta_cols].to_csv(meta_path, index=False)
print(f"Saved bertopic_metadata.csv      — {len(df_clean):,} rows × {len(meta_cols)} cols")

# Sanity check: all three files have the same row count
assert len(df_clean) == sum(1 for _ in open(corpus_path)), "Corpus line count mismatch!"
print("\nRow count assertion passed.")

Saved combined_videos_clean.csv  — 28,000 rows × 42 cols
Saved bertopic_corpus.txt        — 28,000 lines
Saved bertopic_metadata.csv      — 28,000 rows × 12 cols

Row count assertion passed.


In [9]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"Input rows               : 28,037")
print(f"Output rows              : {len(df_clean):,}")
print(f"Rows dropped             : {28037 - len(df_clean):,}")
print(f"Output columns           : {df_clean.shape[1]}")
print()
print("--- Text coverage in bertopic_text ---")
has_desc = df_clean["desc_clean"].ne("")
has_tags = df_clean["tags_clean"].ne("")
print(f"Title + desc + tags      : {(has_desc & has_tags).sum():,} ({(has_desc & has_tags).mean()*100:.1f}%)")
print(f"Title + desc only        : {(has_desc & ~has_tags).sum():,} ({(has_desc & ~has_tags).mean()*100:.1f}%)")
print(f"Title + tags only        : {(~has_desc & has_tags).sum():,} ({(~has_desc & has_tags).mean()*100:.1f}%)")
print(f"Title only               : {(~has_desc & ~has_tags).sum():,} ({(~has_desc & ~has_tags).mean()*100:.1f}%)")
print(f"Sparse (< 5 tokens)      : {df_clean['is_sparse_text'].sum():,} ({df_clean['is_sparse_text'].mean()*100:.1f}%)")
print()
print("--- Shorts ---")
print(f"is_short (≤ 60s)         : {df_clean['is_short'].sum():,} ({df_clean['is_short'].mean()*100:.1f}%)")
print()
print("--- Output files ---")
print("  data/processed/combined_videos_clean.csv")
print("  data/processed/bertopic_corpus.txt")
print("  data/processed/bertopic_metadata.csv")

PREPROCESSING SUMMARY
Input rows               : 28,037
Output rows              : 28,000
Rows dropped             : 37
Output columns           : 42

--- Text coverage in bertopic_text ---
Title + desc + tags      : 19,446 (69.5%)
Title + desc only        : 5,502 (19.7%)
Title + tags only        : 432 (1.5%)
Title only               : 2,620 (9.4%)
Sparse (< 5 tokens)      : 182 (0.7%)

--- Shorts ---
is_short (≤ 60s)         : 6,406 (22.9%)

--- Output files ---
  data/processed/combined_videos_clean.csv
  data/processed/bertopic_corpus.txt
  data/processed/bertopic_metadata.csv


In [10]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import pickle
from pathlib import Path

RANDOM_STATE   = 42
MIN_TOPIC_SIZE = 30      # min videos per topic — tunable
N_GRAM_RANGE   = (1, 2)  # unigrams + bigrams in topic keywords
TOP_N_WORDS    = 10      # keywords shown per topic
MODEL_DIR      = Path("../../outputs/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("BERTopic stack loaded.")
print(f"Corpus size        : {len(df_clean):,} documents")
print(f"Non-sparse docs    : {(~df_clean['is_sparse_text']).sum():,}")
print(f"Min topic size     : {MIN_TOPIC_SIZE}")

BERTopic stack loaded.
Corpus size        : 28,000 documents
Non-sparse docs    : 27,818
Min topic size     : 30


In [12]:
# ── Embedding model ────────────────────────────────────────────────────────
# all-MiniLM-L6-v2: fast, lightweight, strong semantic quality for short text
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ── UMAP: dimensionality reduction ─────────────────────────────────────────
# n_neighbors: local vs global structure balance (15 is BERTopic default)
# n_components: 5 is recommended before HDBSCAN (not 2 — that's for visualization only)
# metric: cosine works well for sentence embeddings
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=False,
)

# ── HDBSCAN: clustering ────────────────────────────────────────────────────
# min_cluster_size = MIN_TOPIC_SIZE controls topic granularity
# prediction_data=True required for transform() on new documents later
hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

# ── Vectorizer: topic keyword extraction ───────────────────────────────────
# Remove English stopwords, use bigrams, ignore very rare/common terms
vectorizer_model = CountVectorizer(
    ngram_range=N_GRAM_RANGE,
    stop_words="english",
    min_df=5,
    max_df=0.85,
)

# ── Representation: refine topic labels beyond c-TF-IDF ───────────────────
representation_model = KeyBERTInspired()

print("All component models built.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

All component models built.


In [13]:
topic_model = BERTopic(
    embedding_model      = embedding_model,
    umap_model           = umap_model,
    hdbscan_model        = hdbscan_model,
    vectorizer_model     = vectorizer_model,
    representation_model = representation_model,
    top_n_words          = TOP_N_WORDS,
    nr_topics            = "auto",   # merge near-duplicate topics post-fit
    calculate_probabilities = False, # saves memory on large corpus
    verbose              = True,
)

# Train on full corpus (sparse docs included — HDBSCAN will assign them to -1)
corpus = df_clean["bertopic_text"].tolist()

print(f"Training on {len(corpus):,} documents...")
topics, _ = topic_model.fit_transform(corpus)

print(f"\nDone. Topics found (excl. outlier -1): {len(set(topics)) - 1}")
print(f"Outlier docs (topic -1)              : {topics.count(-1):,} ({topics.count(-1)/len(topics)*100:.1f}%)")

2026-02-25 13:52:43,657 - BERTopic - Embedding - Transforming documents to embeddings.


Training on 28,000 documents...


Batches:   0%|          | 0/875 [00:00<?, ?it/s]

2026-02-25 13:56:08,295 - BERTopic - Embedding - Completed ✓
2026-02-25 13:56:08,299 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-25 13:56:30,720 - BERTopic - Dimensionality - Completed ✓
2026-02-25 13:56:30,721 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-25 13:56:32,451 - BERTopic - Cluster - Completed ✓
2026-02-25 13:56:32,452 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-02-25 13:56:34,967 - BERTopic - Representation - Completed ✓
2026-02-25 13:56:34,968 - BERTopic - Topic reduction - Reducing number of topics
2026-02-25 13:56:35,017 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-25 13:56:51,812 - BERTopic - Representation - Completed ✓
2026-02-25 13:56:51,828 - BERTopic - Topic reduction - Reduced number of topics from 220 to 119



Done. Topics found (excl. outlier -1): 118
Outlier docs (topic -1)              : 10,459 (37.4%)


In [14]:
# ── Topic info table ───────────────────────────────────────────────────────
topic_info = topic_model.get_topic_info()
print(f"Total topics (incl. -1): {len(topic_info)}")
print()
print(topic_info[topic_info["Topic"] != -1].head(20).to_string(index=False))

# ── Attach topic assignments back to metadata ──────────────────────────────
df_clean["topic_id"] = topics

# ── Per-topic performance stats ────────────────────────────────────────────
topic_stats = (
    df_clean[df_clean["topic_id"] != -1]
    .groupby("topic_id")
    .agg(
        video_count      = ("video_id",       "count"),
        median_views     = ("views",          "median"),
        median_like_rate = ("like_rate",      "median"),
        top_category     = ("category_name",  lambda x: x.value_counts().index[0]),
        category_mix     = ("category_name",  lambda x: x.value_counts().to_dict()),
    )
    .reset_index()
)

# Merge in topic keywords from topic_info
topic_labels = topic_info[topic_info["Topic"] != -1][["Topic", "Name"]].rename(
    columns={"Topic": "topic_id", "Name": "topic_label"}
)
topic_stats = topic_stats.merge(topic_labels, on="topic_id")

# Sort by median views descending
topic_stats = topic_stats.sort_values("median_views", ascending=False).reset_index(drop=True)

print("\n=== Top 15 topics by median views ===")
print(
    topic_stats.head(15)[["topic_id", "topic_label", "video_count", "median_views", "median_like_rate", "top_category"]]
    .to_string(index=False)
)

# Save topic stats
topic_stats.to_csv("../../data/processed/topic_stats.csv", index=False)
print("\nSaved → data/processed/topic_stats.csv")

Total topics (incl. -1): 119

 Topic  Count                                                                                  Name                                                                                                                                                                                              Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [15]:
# ── 1. Outlier rate ────────────────────────────────────────────────────────
outlier_rate = (df_clean["topic_id"] == -1).mean()
print(f"Outlier rate: {outlier_rate*100:.1f}%  (target: < 20%)")
if outlier_rate > 0.20:
    print("  ⚠ High outlier rate — consider lowering MIN_TOPIC_SIZE")
else:
    print("  ✓ Acceptable outlier rate")

# ── 2. Topic size distribution ────────────────────────────────────────────
print(f"\nTopic size distribution (excl. outliers):")
print(topic_stats["video_count"].describe().round(1).to_string())

small_topics = (topic_stats["video_count"] < 50).sum()
print(f"\nTopics with < 50 videos: {small_topics} — consider merging if high")

# ── 3. Category coherence — do topics cluster within categories? ──────────
# A coherent topic should draw heavily from one category
def dominant_share(d):
    vals = list(d.values())
    return max(vals) / sum(vals)

topic_stats["dominant_category_share"] = topic_stats["category_mix"].apply(dominant_share)
print(f"\nMedian dominant-category share per topic: {topic_stats['dominant_category_share'].median():.2f}")
print(f"  (1.0 = topic is purely one category, 0.17 = perfectly mixed)")
print(f"Topics that are >80% one category: {(topic_stats['dominant_category_share'] > 0.8).sum()}")
print(f"Topics that are cross-category    : {(topic_stats['dominant_category_share'] <= 0.5).sum()}")

# ── 4. Spot-check: sample 3 topics ────────────────────────────────────────
print("\n=== Spot-check: 3 random non-outlier topics ===")
sample_topics = topic_stats.sample(3, random_state=RANDOM_STATE)["topic_id"].tolist()
for tid in sample_topics:
    keywords = topic_model.get_topic(tid)
    kw_str = ", ".join([w for w, _ in keywords[:8]])
    examples = df_clean[df_clean["topic_id"] == tid]["video_title"].sample(
        min(3, (df_clean["topic_id"] == tid).sum()), random_state=RANDOM_STATE
    ).tolist()
    print(f"\n  Topic {tid} | Keywords: {kw_str}")
    for ex in examples:
        print(f"    - {ex}")

Outlier rate: 37.4%  (target: < 20%)
  ⚠ High outlier rate — consider lowering MIN_TOPIC_SIZE

Topic size distribution (excl. outliers):
count     118.0
mean      148.7
std       320.0
min        32.0
25%        46.0
50%        54.5
75%        98.0
max      2529.0

Topics with < 50 videos: 41 — consider merging if high

Median dominant-category share per topic: 0.97
  (1.0 = topic is purely one category, 0.17 = perfectly mixed)
Topics that are >80% one category: 105
Topics that are cross-category    : 2

=== Spot-check: 3 random non-outlier topics ===

  Topic 45 | Keywords: fallout season, fallout, trailer fallout, prime video, vegas, season trailer, amazon prime, tv shows
    - Cross S1 Recap | Prime Video
    - FALLOUT SEASON 2 EPISODE 6 Easter Eggs & WTF Ending Ending Explained
    - Is True Detective Worth Watching? | Full Review of All 4 Seasons (No Spoilers)

  Topic 6 | Keywords: verge, tech news, gadget tech, vergecast, future tech, tech gadgets, vergecast youtube, gadget revi

In [16]:
# ── Save BERTopic model ────────────────────────────────────────────────────
model_path = MODEL_DIR / "bertopic_model"
topic_model.save(str(model_path), serialization="pickle", save_ctfidf=True)
print(f"Model saved → {model_path}")

# ── Save enriched metadata (original + topic assignments) ─────────────────
meta_out = df_clean[[
    "video_id", "video_title", "channel_title", "category_name",
    "publish_year", "views", "like_rate", "comment_rate",
    "duration_sec", "is_short", "is_sparse_text",
    "bertopic_token_count", "topic_id",
]].copy()

meta_out = meta_out.merge(
    topic_labels, on="topic_id", how="left"
)
meta_out["topic_label"] = meta_out["topic_label"].fillna("outlier")

meta_path = "../../data/processed/bertopic_metadata.csv"
meta_out.to_csv(meta_path, index=False)
print(f"Enriched metadata saved → {meta_path}  ({len(meta_out):,} rows)")

# ── Quick final summary ────────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"BERTOPIC TRAINING SUMMARY")
print(f"{'='*50}")
print(f"Documents trained on   : {len(corpus):,}")
print(f"Topics found           : {len(set(topics)) - 1}")
print(f"Outlier docs (topic -1): {topics.count(-1):,} ({topics.count(-1)/len(topics)*100:.1f}%)")
print(f"Median topic size      : {topic_stats['video_count'].median():.0f} videos")
print(f"Largest topic          : {topic_stats['video_count'].max()} videos")
print(f"Outputs saved:")
print(f"  outputs/models/bertopic_model")
print(f"  data/processed/topic_stats.csv")
print(f"  data/processed/bertopic_metadata.csv")

2026-02-25 13:59:35,755 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Model saved → ../../outputs/models/bertopic_model
Enriched metadata saved → ../../data/processed/bertopic_metadata.csv  (28,000 rows)

BERTOPIC TRAINING SUMMARY
Documents trained on   : 28,000
Topics found           : 118
Outlier docs (topic -1): 10,459 (37.4%)
Median topic size      : 54 videos
Largest topic          : 2529 videos
Outputs saved:
  outputs/models/bertopic_model
  data/processed/topic_stats.csv
  data/processed/bertopic_metadata.csv
